In [1]:
from utils import *

In [7]:
import mysql.connector

# Connect directly to the learn_mariadb database
conn = mysql.connector.connect(
    host="localhost",      
    user="root",           # change to your MariaDB username
    password="admin",  # change to your MariaDB password
    database="learn_mariadb"
)

cursor = conn.cursor()

# Create table prompt_storage
cursor.execute("""
CREATE TABLE IF NOT EXISTS prompt_storage (
    ID INT AUTO_INCREMENT PRIMARY KEY,
    prompt TEXT NOT NULL,
    pdf_path VARCHAR(500),
    LLM_model VARCHAR(255),
    OCR_model VARCHAR(255),
    create_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    LLM_response TEXT
);
""")

print("Table 'prompt_storage' created successfully!")

# Close connection
cursor.close()
conn.close()


Table 'prompt_storage' created successfully!


In [2]:
test_pdf = 'U1170655.pdf'
mistral_model = "mistral-ocr-latest"
test_response = mistral_ocr_response(test_pdf, mistral_model)
test_response

{'pages': [{'index': 0,
   'markdown': '# Software Product Evaluation Report \n\nReport Title: Software Product Evaluation Report: SoundCloud\nStudent: Tran Minh Anh Nguyen\nStudent ID: U1170655\nSubmission Date: $4^{\\text {th }}$ July 2025\nCourse: CSC1410 - Software Engineering Foundations\nLecturer: Dr. Zhi Chen',
   'images': [],
   'dimensions': {'dpi': 200, 'height': 2200, 'width': 1700}},
  {'index': 1,
   'markdown': 'In recent years, the development of the music industry has been significantly faster. This has led to the strong development of many online music streaming platforms. However, this essay will focus on the one of the popular platforms: SoundCloud application.\n\nFirstly, as a platform that provides open music services, SoundCloud will have basic functions for artists to upload, share and promote music, and will allow users to discover and search for their favorite songs and artists.\n\nSecondly, SoundCloud is designed to meet a wide range of needs, serving both cr

In [3]:
mistral_response = []
for page in test_response.get('pages', []):
    page_number = page.get("index") + 1
    markdown = page.get('markdown', '').strip()
    mistral_response.append(f"\n--- PageNumber {page_number} ---")
    mistral_response.append(markdown)
joined_mistral_response = "\n".join(mistral_response)
print(joined_mistral_response)


--- PageNumber 1 ---
# Software Product Evaluation Report 

Report Title: Software Product Evaluation Report: SoundCloud
Student: Tran Minh Anh Nguyen
Student ID: U1170655
Submission Date: $4^{\text {th }}$ July 2025
Course: CSC1410 - Software Engineering Foundations
Lecturer: Dr. Zhi Chen

--- PageNumber 2 ---
In recent years, the development of the music industry has been significantly faster. This has led to the strong development of many online music streaming platforms. However, this essay will focus on the one of the popular platforms: SoundCloud application.

Firstly, as a platform that provides open music services, SoundCloud will have basic functions for artists to upload, share and promote music, and will allow users to discover and search for their favorite songs and artists.

Secondly, SoundCloud is designed to meet a wide range of needs, serving both creators and consumers of content, with its core audience being young people with $63.5 \%$ of its audience being between t

In [4]:
prompt = "Summarise the content of the PDF document."
LLM_response = LLM_extraction(prompt, test_pdf, model = "gemini-2.5-pro")

token in extracting content
candidates_token_count: 495
Thoughts tokens: 1638
prompt_token_count: 784
total_token_count: 2917
Time taken for LLM extraction: 30.321647882461548 seconds


In [5]:
LLM_response

'Based on the provided document, here is a summary of the "Software Product Evaluation Report: SoundCloud":\n\nThis document is a software product evaluation report on the SoundCloud application, written by student Tran Minh Anh Nguyen for the course CSC1410 - Software Engineering Foundations.\n\nThe report analyzes SoundCloud from a software engineering perspective, making the following key points:\n\n*   **Classification:** SoundCloud is identified as **generic software**, designed for a broad audience of both music creators and consumers, with a core demographic of users aged 18-34.\n\n*   **Key Features:**\n    1.  **Uploading and Sharing:** Its core function allows artists (both famous and unknown) to easily upload and distribute their music.\n    2.  **Timed Comments:** A unique feature that lets users comment on specific moments in a track, fostering deeper community interaction.\n    3.  **Personalized Recommendations:** Uses machine learning to suggest music based on a user\'s

In [14]:
import mysql.connector
from datetime import datetime

# Connect to the learn_mariadb database
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="admin",
    database="learn_mariadb"
)

cursor = conn.cursor()

# Correct INSERT query (excluding ID so it auto-increments)
sql = """
INSERT INTO prompt_storage (prompt, pdf_path, LLM_model, OCR_model, LLM_response, create_time)
VALUES (%s, %s, %s, %s, %s, %s)
"""

values = (
    prompt,                     # variable
    test_pdf,                   # variable
    "gemin-2.5-pro",            # fixed string
    mistral_model,               # variable
    LLM_response,                # variable
    datetime.now().strftime('%Y-%m-%d %H:%M:%S')  # current time
)

cursor.execute(sql, values)
conn.commit()

print(f"{cursor.rowcount} record inserted successfully!")

cursor.close()
conn.close()

1 record inserted successfully!
